### Vector Index

A **vector index** is a data structure that organizes **embedding vectors** so that similar vectors can be found efficiently.

### Why Do We Need Vector Indexes?

In RAG, we may have **thousands or millions of chunk embeddings**. When a query comes in, we need to find the chunks whose vectors are most similar to the query vector.

Without an index, we would have to compare the query against **every stored vector**:

```text
Query
  │
  ├── compare → Vector 1
  ├── compare → Vector 2
  ├── compare → Vector 3
  ├── ...
  └── compare → Vector 1,000,000
```

This becomes slow as the number of vectors increases.

A vector index allows the system to **search through the vectors efficiently and quickly find the most similar ones**.

> **Vector index = a structure that makes searching through large collections of embeddings fast and efficient.**


In [ ]:
%pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 54.8 MB/s eta 0:00:00


FAISS is a library for efficient similarity search and clustering of dense vectors.

In [3]:
import faiss
import numpy as np

In [4]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

documents = [
    "Dogs are loyal domestic animals.",
    "Cats are independent domestic animals.",
    "A CPU executes instructions in a computer.",
    "Python is a programming language.",
    "The Eiffel Tower is located in Paris."
]

# embed docs
document_embeddings = model.encode(documents)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [6]:
# FAISS indexes vectors for efficient nearest-neighbor search.
index = faiss.IndexFlatL2(384)

# Turn embedded docs to np array
vectors = np.array(document_embeddings).astype("float32")

# add them to index
index.add(vectors)

print(index.ntotal)

5


In [7]:
query = "Which animals are good pets?"

# embed the query
query_embedding = model.encode([query])

# turn it to np array
query_vector = np.array(query_embedding).astype("float32")

# search the index, and here k means how many nearest/relevant vectors do I want back?
distances, indices = index.search(query_vector, k=2)

print("Distances:", distances)
print("Indices:", indices)

Distances: [[0.8548665 1.2331481]]
Indices: [[0 1]]


`k` specifies how many nearest vectors we want returned.

For example:
- `k=1` → closest vector
- `k=5` → 5 closest vectors
- `k=10` → 10 closest vectors

In RAG, these nearest vectors correspond to the chunks most relevant
to the user's query.

In [8]:
for distance, idx in zip(distances[0], indices[0]):
    print(f"{distance:.3f}  {documents[idx]}")

0.855  Dogs are loyal domestic animals.
1.233  Cats are independent domestic animals.


### The important concept

We're using:

```python
faiss.IndexFlatL2(384)
```

`L2` means **Euclidean distance**.

So unlike our previous cosine similarity:

```text
higher score → more similar
```

with L2 distance:

```text
lower distance → more similar
```

Conceptually:

```text
query
  │
  ├── document A → distance 0.4  ← closer
  ├── document B → distance 0.8
  └── document C → distance 2.1  ← farther
```

> **Why were we using cosine similarity before, but FAISS is using L2 distance now?**


Because **we chose two different ways of measuring "closeness."**

### Before FAISS

We manually did:

```text
query vector
      ↓
cosine similarity
      ↓
score
```

We used cosine similarity because it's a very common metric for **text embeddings**. It focuses on the **direction** of the vectors, which is useful because the direction represents semantic relationships.

Higher = more similar.

---

### With FAISS

We created:

```python
index = faiss.IndexFlatL2(384)
```

The `L2` explicitly tells FAISS:

> Use **Euclidean distance** to determine which vectors are nearest.

So:

```text
query vector
      ↓
FAISS
      ↓
L2 distance
      ↓
nearest vectors
```

Lower = closer.

---

### But why didn't FAISS just use cosine?

**It can.**

We picked `IndexFlatL2` because it is the simplest vector index to introduce the concept of **nearest-neighbor search**.

FAISS also supports inner-product search, which can be used to implement cosine similarity when the vectors are normalized.

So the distinction is:

```text
Vector search
     │
     ├── Cosine similarity
     ├── L2 distance
     ├── Inner product
     └── other metrics
```

They're all different ways of answering:

> **"Which vectors are closest to my query?"**

### And here's the important bit for our RAG system

We don't necessarily want to casually mix metrics.

If we decide:

> "Our embeddings should be searched using cosine similarity"

then we'd configure the vector index appropriately and use that consistently for both:

```text
documents → embeddings → index
query     → embedding → same metric → search
```

For learning, **L2 is perfectly fine** because it lets you understand what a vector index actually does.

So we haven't made an architectural decision here. We're just learning the machinery.


The next concept is **how a vector index can become faster than checking every vector one by one**.

Right now:

```python
faiss.IndexFlatL2(384)
```

is actually doing a **brute-force search**. It checks the query against every stored vector.

That's fine for our 5 vectors, but imagine:

```text
5 vectors          → trivial
100,000 vectors    → noticeable
10,000,000 vectors → expensive
```

So:

# NEW SUBTOPIC — Approximate Nearest Neighbor (ANN)

The basic idea is:

```text
Exact search:
query
 ↓
compare with EVERY vector
 ↓
guaranteed nearest neighbors
```

versus:

```text
Approximate search:
query
 ↓
use an index/structure to narrow the candidates
 ↓
search only promising vectors
 ↓
very fast nearest neighbors
```

You sacrifice a little exactness for a **huge speed improvement** at scale.

That's the reason vector indexes exist in the first place—not merely to store vectors, but to make **nearest-neighbor retrieval efficient**.


Let's visualize the idea.

Imagine our vectors were only 2D:

```text
                    • B
              •
        •
                  •
   •
                         • C
───────────────┼──────────────────→
        •
             • A
```

With **exact search**, the query has to compare itself against **every single point**:

```text
Query → A
      → B
      → C
      → ...
      → millions of vectors
```

ANN tries to organize the vectors so it can say:

> "The query is around *here*, so I only need to investigate vectors around this region."

Conceptually:

```text
             ┌───────────────┐
             │   region A    │
             │ • • • •       │
             └───────────────┘

                      QUERY
                        ↓
             ┌───────────────┐
             │   region B    │
             │ • • • • •     │
             └───────────────┘

             Ignore most other regions
```

So instead of:

```text
1,000,000 vectors
       ↓
1,000,000 comparisons
```

you might narrow it down to something like:

```text
1,000,000 vectors
       ↓
identify promising region(s)
       ↓
10,000 candidates
       ↓
search those
```

The exact mechanics depend on the **index algorithm**.

And here's the important distinction:

### Exact vs approximate

**Exact nearest-neighbor search**

> "Give me the mathematically closest vectors. I don't care how long it takes."

**Approximate nearest-neighbor search**

> "Give me vectors that are *very likely* to be the closest, but do it much faster."

For RAG, this tradeoff is usually worth it because we don't need the mathematically perfect 1st, 2nd, and 3rd closest vectors—we need **good relevant chunks quickly**.

Next we'll look at one of the simplest ANN approaches used by FAISS: **partitioning the vector space into clusters**. That's where `IndexIVFFlat` comes in.


Let's build the **clustered ANN** idea with FAISS.

### The idea

Instead of treating 1,000,000 vectors as one giant pile, we group nearby vectors into **clusters**.

```text
1,000,000 vectors
       ↓
   clustering
       ↓
┌──────┬──────┬──────┬──────┐
│  C1  │  C2  │  C3  │ ...  │
└──────┴──────┴──────┴──────┘
```

When a query arrives:

```text
query
  ↓
find nearby cluster(s)
  ↓
search vectors inside those clusters
  ↓
top-k results
```

So instead of searching **everything**, we search a much smaller candidate set.

### FAISS calls this IVF

**IVF = Inverted File Index.**

Don't worry about the name yet. The important idea is:

> **Partition the vector space into clusters, then only search selected clusters.**

Now let's actually create one.



In [11]:
import faiss
import numpy as np

vectors = np.array(document_embeddings).astype("float32")

dimension = 384
num_clusters = 2

# IVF partitions vectors into clusters so searches can examine
# only promising regions instead of the entire dataset.
quantizer = faiss.IndexFlatL2(dimension)

index = faiss.IndexIVFFlat(
    quantizer,
    dimension,
    num_clusters,
    faiss.METRIC_L2
)

index.train(vectors)
index.add(vectors)

print(index.ntotal)

5


There's one new thing here:

```python id="qj8h9a"
index.train(vectors)
```

**IVF needs to learn where the clusters should be**, so we train it using our vectors:

```python id="n8x9tb"
index.train(vectors)
index.add(vectors)

print(index.ntotal)
```

You'll get:

```text
5
```

because our five vectors have been added.

### One warning

Our dataset has **only 5 vectors**, so IVF is completely overkill here.

We're using it purely to understand the mechanism.

With a real corpus containing hundreds of thousands or millions of chunks, this kind of indexing becomes much more useful.


In [13]:
query = "Dogs are great pets."

query_embedding = model.encode([query]).astype("float32")

k = 2

distances, indices = index.search(query_embedding, k)

print("Distances:", distances)
print("Indices:", indices)

Distances: [[0.6903733 1.3047483]]
Indices: [[0 1]]


In [14]:
# Search only the single closest IVF cluster.
index.nprobe = 1

distances, indices = index.search(query_embedding, 2)

print("Distances:", distances)
print("Indices:", indices)

Distances: [[0.6903733 1.3047483]]
Indices: [[0 1]]


In [15]:
index.nprobe = 2

distances, indices = index.search(query_embedding, 2)

print("Distances:", distances)
print("Indices:", indices)

Distances: [[0.6903733 1.3047483]]
Indices: [[0 1]]


## `IndexFlatL2` vs `IndexIVFFlat`

The difference is fundamentally **how much of the dataset gets searched**.

### 1. `IndexFlatL2` — exact search

```python
index = faiss.IndexFlatL2(384)
```

Conceptually:

```text
Query
  ↓
compare against
EVERY vector
  ↓
calculate distances
  ↓
sort / select top-k
```

If you have:

```text
1,000,000 vectors
```

FAISS examines all 1,000,000.

So it gives you the **true nearest neighbors**.

That's why it's called **exact search**.

---

### 2. `IndexIVFFlat` — approximate search

```python
index = faiss.IndexIVFFlat(...)
```

First, the vectors are divided into clusters:

```text
1,000,000 vectors
        ↓
     clustering
        ↓
┌────┬────┬────┬────┬────┐
│ C1 │ C2 │ C3 │ C4 │ ...│
└────┴────┴────┴────┴────┘
```

Query:

```text
Query
  ↓
find closest clusters
  ↓
search only those clusters
  ↓
top-k
```

So instead of examining 1,000,000 vectors, perhaps only tens of thousands are examined.

Much faster.

But there's a catch:

**The actual nearest vector might be sitting in a cluster we didn't search.**

That's where recall comes in.

---

## The tradeoff

Think:

```text
                    SEARCH QUALITY
                         ↑
                         │
                  ●      │
             ●           │
        ●                │
   ●                     │
─────────────────────────┴────→ SEARCH COST
```

More searching:

```text
higher recall
higher computation
slower
```

Less searching:

```text
lower recall
lower computation
faster
```

`nprobe` is one of the knobs controlling this tradeoff.

---

## The important mental model

Don't think:

> IVF is a better version of Flat.

Think:

> **They make different tradeoffs.**

|          | `IndexFlatL2`   | `IndexIVFFlat`    |
| -------- | --------------- | ----------------- |
| Search   | Everything      | Selected clusters |
| Exact?   | Yes             | Approximate       |
| Speed    | Slower at scale | Faster at scale   |
| Recall   | 100%            | Can be <100%      |
| Training | No              | Yes               |
| `nprobe` | No              | Yes               |
